# Phase 12 — Anker-Versuch

Der Zielprompt nennt **keinen Ort**: „each service's local name" für Google
Drive, Dropbox und OneDrive. „Local" hat keinen Referenten — das Modell muss
ihn erfinden, und erfindet einen nicht-englischen. Deshalb steht bei L31
`中文名`.

Vier Arme setzen einen Anker **vor** die Köder-Phrase (kausal muss er davor
stehen): keiner / Brazilian / Japanese / official. Schrift-, Anker- und
String-Lesart sagen Verschiedenes vorher.

Selbstversorgend — **frische Runtime**, dann nur diese Zelle. ~25–45 min.

In [ ]:
# === ANKER-VERSUCH: woran haengt die Disposition an Position 43? ===========
# Der erste Tausch-Entwurf brach ab, bevor er Rechenzeit verbrannte, und lieferte
# dabei den Grund: der Zielprompt nennt DREI US-DIENSTE UND KEINEN ORT.
#
#   "...label the column with each service's local name..."
#   Google Drive, Dropbox, OneDrive - lateinische Marken, kein Land, keine Sprache.
#
# "local name" hat hier also gar keinen Referenten. Lokal wo? Das Modell muss den
# Ort selbst ergaenzen - und ergaenzt einen nicht-englischen. Das erklaert, warum
# die Jacobi-Linse bei L31 ausgerechnet '中文名' liest: woertlich "chinesischer
# Name", die Standardfuellung fuer ein unverankertes "local". Und es verbindet die
# beiden Quellen aus Phase 11, die wir bisher getrennt gefuehrt haben: der
# Koeder-Perzept-Fall und der Anker-Mangel-Fall ("Help me", "TensorFlow")
# koennten dasselbe sein.
#
# Ein Detail erzwingt den Aufbau: Attention ist kausal. Der Zustand an Position 43
# haengt NUR von den Tokens 0..43 ab, jede Ergaenzung danach liesse H[43] bitgleich
# und der Versuch waere per Konstruktion null. Der Anker muss davor.
#   original  "each service's local name"            kein Anker
#   latein    "each service's Brazilian local name"  Anker, lateinische Schrift
#   fremd     "each service's Japanese local name"   Anker, fremde Schrift
#   neutral   "each service's official local name"   Adjektiv, KEIN Ort
# Der neutrale Arm faengt ab, dass schon irgendein zusaetzliches Wort den Effekt
# bricht. Alles VOR dem Koeder bleibt in allen Armen zeichengleich, die Koeder-
# Tokens selbst sind identisch (die Zelle prueft es und bricht sonst ab).
#
# Die drei Lesarten sagen Verschiedenes vorher - das ist der Sinn der Sache:
#   Schrift : latein bricht ein, fremd nicht.
#   Anker   : BEIDE verankerten Arme brechen ein, neutral nicht.
#   String  : nichts bricht ein, die Zeichenkette allein traegt.
#
# Vorregistriert: Ausleseschicht L35 (tiefste, ohne Blick auf Q gewaehlt), L31 und
# L27 sekundaer. Zielgroesse Q/Median DERSELBEN Schicht und desselben Arms, pro
# Korpus-Prompt einzeln; der Korpus-Prompt ist die Einheit fuer den gepaarten
# Bootstrap. Absolute Massen sind zwischen Armen nicht vergleichbar.
#
# Vorregistrierung Nr. 30: ANKER ~40%, SCHRIFT ~25%, UNKLAR ~20%,
# JEDE-ERGAENZUNG ~10%, STRING ~5%.
# Vier Arme statt drei, geschaetzt 25-45 min. FRISCHE Runtime, einzige Zelle.

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
# ---------------- Selbstversorgung: Modell + Prompts sicherstellen ----------
import glob, json, gc
for _n in ("model_b","tok_b"):                  # Base-Reste aus Cell 28 raus
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig): vermutlich "
        "belegt noch ein frueheres Modell den Speicher. Loesung: Laufzeit -> "
        "Sitzung neu starten, dann NUR diese Zelle ausfuehren.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","lauf")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import subprocess, sys, time
N_FIT=12; MAXTOK=64; BATCH=6; SEED=0; EPS=0.1; NBOOT=4000
LAYERS=[27,31,35]           # nur die Schichten, in denen der Effekt lebt
L_PRIM=35                   # VORREGISTRIERT: tiefste Schicht, ohne Blick auf Q gewaehlt
N_CTRL=15
MASK_NPZ=(glob.glob("/content/drive/MyDrive/**/vocab_foreign_masks.npz",recursive=True) or [""])[0]
SCAFF="<|im_start|>user\n"
def think_prefix(u,th=""):
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
# ---------------- reine Logik (offline geprueft) ---------------------------
# Der Zielprompt nennt drei US-Dienste und KEINEN Ort. "local name" hat also
# keinen Referenten - lokal wo? Das Modell muss den Ort selbst ergaenzen. Statt
# einen Locale-Hinweis zu tauschen (es gibt keinen), setzen wir einen VOR die
# Koeder-Phrase. Attention ist kausal: alles nach Position 43 laesst H[43]
# bitgleich, der Anker muss davor stehen.
#   original  "each service's local name"            - kein Anker
#   latein    "each service's Brazilian local name"  - Anker, lateinische Schrift
#   fremd     "each service's Japanese local name"   - Anker, fremde Schrift
#   neutral   "each service's official local name"   - Adjektiv, KEIN Ort
# Der neutrale Arm kontrolliert, ob schon irgendein zusaetzliches Adjektiv den
# Effekt bricht. Die drei Lesarten sagen Verschiedenes vorher:
#   Schrift-Lesart : latein bricht ein, fremd nicht.
#   Anker-Lesart   : BEIDE verankerten Arme brechen ein, neutral nicht.
#   String-Lesart  : nichts bricht ein, die Zeichenkette entscheidet.
ADJEKTIV={"original":"","latein":"Brazilian ","fremd":"Japanese ","neutral":"official "}
def setze_anker(text,adj):
    """schiebt das Adjektiv unmittelbar vor 'local name'; laesst alles davor
       unberuehrt und die Koeder-Phrase zusammenhaengend"""
    if not adj: return text,True
    i=text.find("local name")
    if i<0: return text,False
    return text[:i]+adj+text[i:],True
def sample_positions(lo,hi,n,must):
    if hi<=lo: return sorted(set(must))
    step=max(1,(hi-lo)//max(1,n))
    return sorted(set(list(range(lo,hi+1,step))[:n]+[t for t in must if lo<=t<=hi]))
def chunks(xs,k): return [xs[i:i+k] for i in range(0,len(xs),k)]
def rang(vals,idx):
    v=list(vals); t=v[idx]; return 1+sum(1 for x in v if x>t)
def boot_diff(a,b,nboot=4000,seed=0):
    """gepaarter Bootstrap ueber Korpus-Prompts auf log10(a)-log10(b)"""
    a=np.asarray(a,float); b=np.asarray(b,float); n=len(a)
    d=np.log10(np.maximum(a,1e-12))-np.log10(np.maximum(b,1e-12))
    rng=np.random.default_rng(seed)
    bs=np.array([d[rng.integers(0,n,n)].mean() for _ in range(nboot)])
    return float(d.mean()),float(np.percentile(bs,2.5)),float(np.percentile(bs,97.5))
def verdict_anker(lat,fre,neu,schwelle=0.15):
    """je Arm ein Tripel (mittlere log10-Differenz zum Original, CI-unten, CI-oben).
       bricht = Konfidenzintervall komplett unter 0."""
    bricht=lambda t: t[2]<0
    flach =lambda t: t[1]>-schwelle and t[2]<schwelle
    if bricht(neu) and bricht(lat) and bricht(fre): return "JEDE-ERGAENZUNG"
    if bricht(lat) and not bricht(fre): return "SCHRIFT"
    if bricht(lat) and bricht(fre) and not bricht(neu): return "ANKER"
    if flach(lat) and flach(fre) and not bricht(neu): return "STRING"
    return "UNKLAR"
# ---------------- Umgebung sicherstellen -----------------------------------
JL="/content/jacobian_lens"
if "fd_transport" not in globals():
    if not os.path.isdir(os.path.join(JL,"jlens")):
        r=subprocess.run(["git","clone","--depth","1",
                          "https://github.com/Erikiss/jacobian-lens",JL],
                         capture_output=True,text=True,timeout=600)
        assert r.returncode==0, "Klon fehlgeschlagen"
    subprocess.run([sys.executable,"-m","pip","install","-q","--no-deps","-e",JL],
                   capture_output=True,text=True)
    if JL not in sys.path: sys.path.insert(0,JL)
from jlens.hooks import ActivationRecorder
from jlens.fitting import valid_position_mask
BLOCKS=model.model.layers
for _p in model.parameters(): _p.requires_grad_(False)
def fwd(ids): return model.model(input_ids=ids)
_UD=next(model.model.norm.parameters()).dtype
def unembed(r): return model.lm_head(model.model.norm(r.to(_UD)))
@torch.no_grad()
def fd_transport(input_ids,source_layers,target_layer,V,skip_first,B,eps=EPS):
    """zentrale Differenz - kein Autograd, laeuft durch jeden Kernel"""
    ids=input_ids.expand(B,-1)
    pm=valid_position_mask(ids.shape[1],skip_first=skip_first); npos=int(pm.sum())
    st={"l":None,"sign":0,"tan":None,"pos":pm.nonzero(as_tuple=True)[0]}
    def mk(idx):
        def hook(mod,inp,out):
            if st["l"]!=idx or st["sign"]==0: return None
            t=out if torch.is_tensor(out) else out[0]
            t2=t.clone(); p=st["pos"].to(t2.device)
            t2[:,p,:]=t2[:,p,:]+(st["sign"]*eps)*st["tan"].to(t2.dtype).to(t2.device)[:,None,:]
            return t2 if torch.is_tensor(out) else (t2,)+tuple(out[1:])
        return hook
    hs=[BLOCKS[l].register_forward_hook(mk(l)) for l in source_layers]
    out={}
    try:
        with ActivationRecorder(BLOCKS,at=[target_layer]) as rec:
            def run():
                fwd(ids); a=rec.activations[target_layer]
                return a[:,st["pos"].to(a.device),:].float().sum(1)
            for l in source_layers:
                st["l"]=l; st["tan"]=V[l]
                st["sign"]=1; yp=run(); st["sign"]=-1; ym=run()
                st["l"]=None; st["sign"]=0
                out[l]=(yp-ym)/(2.0*eps*npos)
    finally:
        for h in hs: h.remove()
    return out
# ---------------- Drive-Ausgabe --------------------------------------------
OUT=globals().get("RUN_OUT") or ("/content/drive/MyDrive/WeirdChat_Runs/tausch_"
                                 +time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(OUT,exist_ok=True)
LINES=[]
def P(s=""): LINES.append(str(s))
def schreibe(n,t):
    with open(os.path.join(OUT,n),"w",encoding="utf-8") as f: f.write(t)
FEHLER=None
try:
    TAB=PROMPTS[[p for p in PROMPT_IDS if p.startswith("643fdf5d")][0]]
    P("ANKER-VERSUCH - %s"%time.strftime("%Y-%m-%d %H:%M:%S")); P("="*72)
    P("ZIELPROMPT (%d Zeichen):"%len(TAB)); P(TAB); P("")
    assert "local name" in TAB, "Koeder-Phrase nicht im Prompt"
    ARME={}
    for a,adj in ADJEKTIV.items():
        t,ok=setze_anker(TAB,adj)
        assert ok, "Anker liess sich fuer Arm %s nicht setzen"%a
        ARME[a]=t
    D={}
    P("AUSRICHTUNG je Arm:")
    for a,txt in ARME.items():
        pre=think_prefix(txt); e=tokenizer(pre,return_offsets_mapping=True)
        ids=e["input_ids"]; om=e["offset_mapping"]
        c0=len(SCAFF)+txt.index("local name"); c1=c0+len("local name")
        dec=[i for i,(x,y) in enumerate(om) if y>c0 and x<c1 and y>x]
        K,Q=dec[0],dec[-1]
        ut=[i for i,(x,y) in enumerate(om) if y>len(SCAFF) and x<len(SCAFF)+len(txt) and y>x]
        pos=sample_positions(ut[0],ut[-1],N_CTRL,[K-1,K,Q,Q+1])
        D[a]=dict(text=txt,ids=ids,K=K,Q=Q,POS=pos,jQ=pos.index(Q),L=len(ids),
                  davor=tokenizer.decode([ids[K-1]]))
        P("  %-9s %3d Tok | vor dem Koeder %r | K=%d %r Q=%d %r | %d Pos"
          %(a,len(ids),D[a]["davor"],K,tokenizer.decode([ids[K]]),
            Q,tokenizer.decode([ids[Q]]),len(pos)))
    gl=len({(tokenizer.decode([D[a]["ids"][D[a]["K"]]]),
             tokenizer.decode([D[a]["ids"][D[a]["Q"]]])) for a in ARME})==1
    P("  Koeder-Tokens in allen Armen identisch: %s"%gl)
    assert gl, "Koeder-Tokens unterscheiden sich - der Test waere konfundiert"
    P("  (Positionen verschieben sich um ein Token; Raenge sind arminterne")
    P("   Vergleiche, das ist unproblematisch.)")
    for a in ARME:
        it=torch.tensor([D[a]["ids"]],device=model.device)
        with torch.no_grad():
            hs=model(input_ids=it,output_hidden_states=True).hidden_states
        D[a]["H"]={l:torch.stack([hs[l+1][0,p] for p in D[a]["POS"]]).float().cpu()
                   for l in LAYERS}
        del hs
    gc.collect(); torch.cuda.empty_cache()
    rng=np.random.default_rng(SEED)
    cand=[p for p in PROMPT_IDS if 200<len(PROMPTS[p])<=1200]
    corp=[PROMPTS[cand[i]] for i in rng.permutation(len(cand))[:N_FIT]]
    P(""); P("Korpus: %d Prompts (gemeinsam fuer alle Arme), je bis %d Tokens"
             %(len(corp),MAXTOK))
    TGT=model.config.num_hidden_layers-1
    M_script=torch.tensor(np.load(MASK_NPZ)["script"])
    def fmass(lg):
        p=torch.softmax(lg.float(),-1); V=p.shape[-1]
        m=M_script.to(p.device)
        if m.shape[0]<V: m=torch.cat([m,torch.zeros(V-m.shape[0],dtype=torch.bool,device=m.device)])
        return p[...,m[:V]].sum(-1)
    RES={a:{l:[] for l in LAYERS} for a in ARME}
    t0=time.time(); nok=0
    for ci,ctext in enumerate(corp):
        cid=tokenizer(ctext,return_tensors="pt",truncation=True,
                      max_length=MAXTOK).input_ids.to(model.device)
        if cid.shape[1]<24: continue
        try:
            for a in ARME:
                pos=D[a]["POS"]; H=D[a]["H"]; jQ=D[a]["jQ"]
                acc={l:torch.zeros(len(pos),model.config.hidden_size) for l in LAYERS}
                for g in chunks(list(range(len(pos))),BATCH):
                    V={l:H[l][g].to(model.device) for l in LAYERS}
                    o=fd_transport(cid,LAYERS,TGT,V,16,len(g))
                    for l in LAYERS: acc[l][g]=o[l].cpu()
                    del o,V
                with torch.no_grad():
                    for l in LAYERS:
                        m=fmass(unembed(acc[l].to(model.device))).cpu().numpy()
                        RES[a][l].append((float(m[jQ]/max(np.median(m),1e-12)),rang(m,jQ)))
            nok+=1
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); P("  OOM bei Korpus-Prompt %d - uebersprungen"%ci)
        gc.collect(); torch.cuda.empty_cache()
        if (ci+1)%3==0:
            P("  %2d/%d | %.1f min | %.1f s je Prompt"
              %(ci+1,len(corp),(time.time()-t0)/60,(time.time()-t0)/max(ci+1,1)))
    assert nok>=4, "zu wenige Korpus-Prompts (%d)"%nok
    P("  verwertbar: %d Korpus-Prompts"%nok)
    P(""); P("ERGEBNIS je Arm und Schicht (n=%d)"%nok)
    P("  %-9s %-6s %15s %10s %12s"%("Arm","L","Q/Median (Med)","Rang-1","Rang (Med)"))
    ST={}
    for a in ARME:
        for l in LAYERS:
            r=RES[a][l]; ratio=[x[0] for x in r]; rk=[x[1] for x in r]
            ST[(a,l)]=dict(ratio=ratio,rank=rk)
            P("  %-9s L%-5d %15.2f %6d/%-3d %12.1f"
              %(a,l,float(np.median(ratio)),sum(1 for x in rk if x==1),len(rk),
                float(np.median(rk))))
    P(""); P("GEPAARTER BOOTSTRAP gegen 'original', vorregistrierte Schicht L%d:"%L_PRIM)
    o=ST[("original",L_PRIM)]["ratio"]; B={}
    for a in ("latein","fremd","neutral"):
        B[a]=boot_diff(ST[(a,L_PRIM)]["ratio"],o,NBOOT,SEED)
        P("  %-8s %+.3f Dekaden  [%+.3f, %+.3f]"%(a,B[a][0],B[a][1],B[a][2]))
    for l in LAYERS:
        if l==L_PRIM: continue
        o2=ST[("original",l)]["ratio"]
        P("  (L%d sekundaer) %s"%(l," | ".join("%s %+.2f [%+.2f,%+.2f]"
          %((a,)+boot_diff(ST[(a,l)]["ratio"],o2,NBOOT,SEED)) for a in ("latein","fremd","neutral"))))
    code=verdict_anker(B["latein"],B["fremd"],B["neutral"])
    P(""); P("VERDIKT: %s"%code)
    if code=="SCHRIFT":
        P("  Ein lateinschriftlicher Anker bricht den Effekt (%+.2f), ein fremd-"%B["latein"][0])
        P("  schriftlicher nicht (%+.2f). Es haengt an der geforderten SCHRIFT."%B["fremd"][0])
        P("  Die Lokalisierungs-Lesart ist ausgeschlossen.")
    elif code=="ANKER":
        P("  BEIDE Anker brechen den Effekt (latein %+.2f, fremd %+.2f), ein blosses"
          %(B["latein"][0],B["fremd"][0]))
        P("  Adjektiv ohne Ort nicht (%+.2f). Nicht die Schrift entscheidet,"%B["neutral"][0])
        P("  sondern dass 'local' UEBERHAUPT einen Referenten bekommt. Das ist")
        P("  die Anker-Mangel-Lesart - und sie vereint die beiden Quellen aus")
        P("  Phase 11 zu einer.")
    elif code=="STRING":
        P("  Kein Anker bewegt etwas (latein %+.2f, fremd %+.2f). Was an Q steht,"
          %(B["latein"][0],B["fremd"][0]))
        P("  haengt an der Zeichenkette 'local name' selbst - die Lokalisierungs-")
        P("  Lesart bleibt stehen.")
    elif code=="JEDE-ERGAENZUNG":
        P("  Jede Ergaenzung bricht den Effekt, auch das ortlose Adjektiv (%+.2f)."%B["neutral"][0])
        P("  Der Test trennt nicht: schon die Stoerung der Wortfolge genuegt.")
    else:
        P("  Kein klares Bild: latein %+.2f [%+.2f,%+.2f], fremd %+.2f, neutral %+.2f."
          %(B["latein"][0],B["latein"][1],B["latein"][2],B["fremd"][0],B["neutral"][0]))
        P("  Mehr Korpus-Prompts oder ein Effekt mittlerer Groesse.")
    P(""); P("(Zielgroesse ist Q/Median DERSELBEN Schicht und desselben Arms.")
    P(" Absolute Massen sind zwischen Armen nicht vergleichbar.)")
    fig,axs=plt.subplots(1,len(LAYERS),figsize=(4.8*len(LAYERS),4.2))
    if len(LAYERS)==1: axs=[axs]
    for ax,l in zip(axs,LAYERS):
        dat=[np.log10(np.maximum(ST[(a,l)]["ratio"],1e-12)) for a in ARME]
        ax.boxplot(dat,labels=list(ARME))
        for i,d in enumerate(dat):
            ax.scatter(np.full(len(d),i+1)+np.linspace(-.08,.08,len(d)),d,s=14,
                       color="#DC2626",zorder=3)
        ax.axhline(0,ls="--",c="#888",lw=1)
        ax.set_title("L%d%s"%(l," (vorregistriert)" if l==L_PRIM else ""),fontsize=10)
        ax.set_ylabel("log10(Q / Median)"); ax.tick_params(axis="x",labelrotation=20)
    fig.tight_layout(); fig.savefig(os.path.join(OUT,"anker.png"),dpi=150,bbox_inches="tight")
    TAUSCH_RESULTS=dict(verdict=code,n_corpus=nok,layers=LAYERS,L_prim=L_PRIM,
                        prompts={a:D[a]["text"] for a in ARME},
                        Q={a:D[a]["Q"] for a in ARME},
                        stats={"%s_L%d"%(a,l):ST[(a,l)] for a in ARME for l in LAYERS},
                        boot={a:list(B[a]) for a in B})
    globals()["TAUSCH_RESULTS"]=TAUSCH_RESULTS
    schreibe("ANKER_RESULTS.json",json.dumps(TAUSCH_RESULTS,ensure_ascii=False,indent=1))

except SystemExit:
    pass
except Exception as e:
    import traceback; FEHLER=traceback.format_exc(); P(""); P("ABBRUCH: %s"%e); P(FEHLER)
finally:
    schreibe("bericht_tausch.txt","\n".join(LINES))
    print("GESCHRIEBEN NACH:",OUT)
    print("\n".join(LINES[-45:]) if not FEHLER else FEHLER.splitlines()[-1])
wc_save_all()
